NB1

### 1. Open PowerShell

### 2. Launch WSL
```powershell
wsl
```

### 3. Activate virtual environment
```bash
source ~/vllm_env/bin/activate
```

### 4. Ensure cache is on D drive (optional check)
```bash
echo $HF_HOME
```
Expected:
```bash
/mnt/d/huggingface_cache
```

### 5. Run vLLM server
```bash
python -m vllm.entrypoints.openai.api_server   --model Qwen/Qwen2.5-VL-3B-Instruct   --trust-remote-code   --dtype float16   --quantization bitsandbytes   --load-format bitsandbytes   --max-model-len 6144   --max-num-seqs 2   --gpu-memory-utilization 0.72   --limit-mm-per-prompt '{"image": 1}'   --mm-processor-kwargs '{"min_pixels": 200704, "max_pixels": 1254400}'   --port 8000   --host 0.0.0.0
```

### 6. Wait for server start
Look for:
```text
Uvicorn running on http://0.0.0.0:8000
```

### 7. Verify server (new PowerShell window)
```powershell
curl http://localhost:8000/v1/models
```



In [1]:
# Cell: package install
import subprocess, sys

packages = [
    "pdfplumber",
    "pymupdf",
    "Pillow",
    "tqdm",
    "orjson",
    "openai",      # vLLM speaks OpenAI API
    "requests",
    "aiohttp",     # required by async client
    "nest_asyncio"
]

for pkg in packages:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    print(f"  {'OK' if r.returncode == 0 else 'FAIL'} {pkg}")

print("\nDone.")

  OK pdfplumber
  OK pymupdf
  OK Pillow
  OK tqdm
  OK orjson
  OK openai
  OK requests
  OK aiohttp
  OK nest_asyncio

Done.


In [2]:
# Cell: Config (replace entire cell)
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Paths
RESUMES_DIR    = "../data/raw/resumes"
OUTPUT_DIR     = "../data/extracted_text"
ENRICHED_JSONL = f"{OUTPUT_DIR}/enriched_resumes.jsonl"
PROGRESS_FILE  = f"{OUTPUT_DIR}/ingestion_progress.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# vLLM server
VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_MODEL    = "Qwen/Qwen2.5-VL-3B-Instruct"

# Stability + speed balance
CONCURRENT_REQUESTS = 2          # keep GPU fed
THREAD_WORKERS      = 1          # reduce native PDF memory pressure
CHECKPOINT_EVERY    = 10
CHUNK_SIZE          = 150        # recreate session/executor every chunk to prevent long-run leaks

TEXT_MIN_CHARS      = 50
RENDER_DPI          = 170        # lower memory footprint than 180
MAX_RENDER_PIXELS   = 3_500_000  # hard cap on render size
JPEG_QUALITY        = 82

REQUEST_TIMEOUT_S   = 120
RENDER_TIMEOUT_S    = 35
PDF_TASK_TIMEOUT_S  = 220

VLLM_MAX_RETRIES    = 1
RETRY_BACKOFF_S     = 1.5
HEALTH_TIMEOUT_S    = 5
COOLDOWN_EVERY_PDFS = 100
COOLDOWN_SECONDS = 30
# Must align with server range
MM_MIN_PIXELS = 256 * 28 * 28
MM_MAX_PIXELS = 1600 * 28 * 28

MAX_NEW_TOKENS = 1600

print("Config ready.")
print(f"  vLLM URL         : {VLLM_BASE_URL}")
print(f"  Concurrent PDFs  : {CONCURRENT_REQUESTS}")
print(f"  Thread workers   : {THREAD_WORKERS}")
print(f"  Chunk size       : {CHUNK_SIZE}")
print(f"  Render DPI       : {RENDER_DPI}")
print(f"  Render pixel cap : {MAX_RENDER_PIXELS:,}")

Config ready.
  vLLM URL         : http://localhost:8000/v1
  Concurrent PDFs  : 2
  Thread workers   : 1
  Chunk size       : 150
  Render DPI       : 170
  Render pixel cap : 3,500,000


In [ ]:
#Verify vLLM server is running
import requests

try:
    r = requests.get(f"{VLLM_BASE_URL.replace('/v1','')}/health", timeout=5)
    print(f"vLLM server: Running ({r.status_code})")
except Exception as e:
    print(f"vLLM server: ✗ — {e}")
    print("Start the server in WSL2 first (see setup instructions)")
    raise

# Also check model is loaded
r = requests.get(f"{VLLM_BASE_URL}/models", timeout=5)
models = r.json()
print(f"Loaded model: {models['data'][0]['id']}")

vLLM server: ✓ (200)
Loaded model: Qwen/Qwen2.5-VL-3B-Instruct


In [4]:
#Discover PDFs
from collections import Counter

root     = Path(RESUMES_DIR)
pdf_list = sorted(root.rglob("*.pdf")) + sorted(root.rglob("*.PDF"))

if not pdf_list:
    raise FileNotFoundError(f"No PDFs found under {RESUMES_DIR}")

manifest = [
    {
        "candidate_id" : str(i + 1).zfill(4),
        "pdf_path"     : str(p),
        "pdf_name"     : p.name,
        "pdf_stem"     : p.stem,
        "category"     : p.parent.name,
    }
    for i, p in enumerate(pdf_list)
]

cat_counts = Counter(m["category"] for m in manifest)
print(f"Total PDFs : {len(manifest)}")
print(f"Categories : {len(cat_counts)}")
for cat, n in sorted(cat_counts.items()):
    print(f"  {cat:45s} {n:>6}")

Total PDFs : 2772
Categories : 43
  Accountant resumes                                88
  Advocate resumes                                 166
  Agricultural resumes                              72
  Apparel resumes                                   92
  Architects resumes                                76
  Arts resumes                                      88
  Automobile resumes                                64
  Aviation resumes                                  78
  BPO resumes                                       30
  Banking resumes                                   52
  Blockchain resumes                                 6
  Building _Construction resumes                    72
  Business Analyst resumes                          62
  Civil Engineer resumes                            78
  Consultant resumes                                88
  Database resumes                                  82
  Designing resumes                                 48
  DevOps Engineer resumes      

In [5]:
#Extraction Functions
import base64
import gc
import io
import pdfplumber
import fitz
from PIL import Image


def clean_text(raw: str) -> str:
    if not raw:
        return ""
    lines = [
        ln.strip() for ln in raw.splitlines()
        if len(ln.strip()) > 1 and sum(c.isalnum() for c in ln) >= 2
    ]
    return "\n".join(lines)


def structure_pdfplumber_text(raw: str) -> str:
    if not raw:
        return ""
    out_lines = []
    for line in raw.splitlines():
        stripped = line.strip()
        if not stripped:
            out_lines.append("")
            continue
        is_heading = (
            stripped.isupper()
            and 2 <= len(stripped.split()) <= 6
            and len(stripped) < 60
        )
        if is_heading:
            out_lines.append(f"\n## {stripped.title()}\n")
        elif stripped.startswith(("•", "·", "▪", "◦", "-", "–", "—")):
            out_lines.append(f"- {stripped[1:].strip()}")
        else:
            out_lines.append(stripped)
    return "\n".join(out_lines).strip()


def extract_digital_pages(pdf_path: str) -> list[dict]:
    results = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                try:
                    raw = page.extract_text(x_tolerance=2, y_tolerance=3) or ""
                    text = structure_pdfplumber_text(raw)
                except Exception:
                    text = ""

                needs_vllm = len(text) < TEXT_MIN_CHARS
                results.append({
                    "page_num": i,
                    "text": text if not needs_vllm else "",
                    "char_count": len(text),
                    "needs_vllm": needs_vllm,
                    "method": "pdfplumber" if not needs_vllm else "pending",
                })
    except Exception as e:
        results = [{
            "page_num": 0,
            "text": "",
            "char_count": 0,
            "needs_vllm": True,
            "method": "pending",
            "error": str(e)
        }]
    return results


def render_page_to_base64(pdf_path: str, page_num: int, dpi: int = RENDER_DPI) -> str | None:
    """
    Memory-bounded page render:
    - dynamically lowers DPI for huge pages
    - grayscale JPEG to reduce RAM + payload size
    """
    try:
        with fitz.open(pdf_path) as doc:
            page = doc[page_num]
            rect = page.rect

            # Estimate output size and clamp DPI if needed
            est_w = max(1, int(rect.width * dpi / 72.0))
            est_h = max(1, int(rect.height * dpi / 72.0))
            est_pixels = est_w * est_h
            if est_pixels > MAX_RENDER_PIXELS:
                scale = (MAX_RENDER_PIXELS / est_pixels) ** 0.5
                dpi = max(110, int(dpi * scale))

            mat = fitz.Matrix(dpi / 72, dpi / 72)
            pix = page.get_pixmap(matrix=mat, alpha=False, colorspace=fitz.csGRAY)

        img = Image.frombytes("L", [pix.width, pix.height], pix.samples)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=JPEG_QUALITY, optimize=True)

        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

        # Explicit cleanup
        img.close()
        buf.close()
        del pix
        fitz.TOOLS.store_shrink(100)
        gc.collect()

        return b64
    except Exception as e:
        print(f"  Render error {pdf_path} page {page_num}: {e}")
        fitz.TOOLS.store_shrink(100)
        gc.collect()
        return None


VLLM_PROMPT = """You are doing OCR transcription of a resume page.

Return only text visibly present in the image.

Rules:
1. Do not invent names, companies, emails, skills, dates, or sections.
2. Do not summarize or paraphrase.
3. If the layout has two columns, read LEFT column top-to-bottom first, then RIGHT column top-to-bottom.
4. Preserve headings, bullets, and line breaks as much as possible.
5. Do not output markdown tables unless the source is an actual table.
6. Keep original language.
7. Output plain markdown text only. No explanations.
8. If unreadable, output exactly: [UNREADABLE_PAGE]
"""


def parse_vllm_output(raw: str) -> str:
    if not raw or not raw.strip():
        return ""
    text = raw.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:])
    if text.endswith("```"):
        text = "\n".join(text.split("\n")[:-1])
    return text.strip()


print("Extraction functions defined")

Extraction functions defined


In [ ]:
#Async LLM Caller
#Parallel processing + API calls + memory control + fault tolerance
import asyncio
import aiohttp
import orjson
import time
import gc
import fitz
from datetime import datetime, timezone
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# Cooldown controls (uses config values if you define them there, else defaults)
COOLDOWN_EVERY_PDFS = globals().get("COOLDOWN_EVERY_PDFS", 75)
COOLDOWN_SECONDS = globals().get("COOLDOWN_SECONDS", 60)


def chunked(items: list, size: int):
    for i in range(0, len(items), size):
        yield items[i:i + size]


async def vllm_healthcheck(session: aiohttp.ClientSession) -> bool:
    base = VLLM_BASE_URL.replace("/v1", "")
    try:
        async with session.get(
            f"{base}/health",
            timeout=aiohttp.ClientTimeout(total=HEALTH_TIMEOUT_S),
        ) as r:
            return r.status == 200
    except Exception:
        return False


async def call_vllm_async(session: aiohttp.ClientSession, b64_image: str) -> str:
    payload = {
        "model": VLLM_MODEL,
        "max_tokens": MAX_NEW_TOKENS,
        "temperature": 0.0,
        "messages": [{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{b64_image}",
                        "min_pixels": MM_MIN_PIXELS,
                        "max_pixels": MM_MAX_PIXELS,
                    },
                },
                {
                    "type": "text",
                    "text": VLLM_PROMPT,
                },
            ],
        }],
    }

    last_err = None
    for attempt in range(VLLM_MAX_RETRIES + 1):
        try:
            async with session.post(
                f"{VLLM_BASE_URL}/chat/completions",
                json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_S),
            ) as resp:
                if resp.status == 200:
                    data = await resp.json()
                    raw = data["choices"][0]["message"]["content"]
                    return parse_vllm_output(raw)

                body = await resp.text()
                last_err = RuntimeError(f"vLLM HTTP {resp.status}: {body[:300]}")
        except Exception as e:
            last_err = e

        if attempt < VLLM_MAX_RETRIES:
            await asyncio.sleep(RETRY_BACKOFF_S * (attempt + 1))

    raise RuntimeError(f"vLLM call failed: {last_err}")


def build_record(meta: dict, page_results: list[dict], elapsed_s: float) -> dict:
    num_pages = len(page_results)
    page_blocks = []
    for p in page_results:
        t = p["text"].strip()
        if t:
            page_blocks.append(
                f"<!-- page {p['page_num'] + 1} -->\n{t}" if num_pages > 1 else t
            )

    content = "\n\n---\n\n".join(page_blocks)
    methods_used = sorted(list({
        p["method"] for p in page_results if p["method"] not in ("pending", "")
    }))
    qwen_pages = sum(1 for p in page_results if p.get("method") == "vllm")

    return {
        "candidate_id": meta["candidate_id"],
        "metadata": {
            "pdf_name": meta["pdf_name"],
            "pdf_stem": meta["pdf_stem"],
            "pdf_path": meta["pdf_path"],
            "category": meta["category"],
            "page_count": num_pages,
            "digital_pages": num_pages - qwen_pages,
            "vllm_pages": qwen_pages,
            "total_chars": len(content),
            "methods_used": methods_used,
            "elapsed_s": round(elapsed_s, 2),
            "processed_at": datetime.now(timezone.utc).isoformat(),
        },
        "content": content,
    }

async def process_pdf_async(
    meta: dict,
    session: aiohttp.ClientSession,
    executor: ThreadPoolExecutor,
    loop: asyncio.AbstractEventLoop,
) -> dict:
    t0 = time.time()

    page_results = await loop.run_in_executor(
        executor, extract_digital_pages, meta["pdf_path"]
    )

    for p in page_results:
        if not p["needs_vllm"]:
            continue

        try:
            b64 = await asyncio.wait_for(
                loop.run_in_executor(
                    executor,
                    render_page_to_base64,
                    meta["pdf_path"],
                    p["page_num"],
                    RENDER_DPI,
                ),
                timeout=RENDER_TIMEOUT_S,
            )
        except asyncio.TimeoutError:
            raise RuntimeError(f"Render timeout: {meta['pdf_name']} page {p['page_num'] + 1}")

        if not b64:
            raise RuntimeError(f"Render failed: {meta['pdf_name']} page {p['page_num'] + 1}")

        txt = await call_vllm_async(session, b64)
        if not txt.strip():
            raise RuntimeError(f"Empty OCR: {meta['pdf_name']} page {p['page_num'] + 1}")

        p["text"] = txt
        p["char_count"] = len(txt)
        p["method"] = "vllm"

        del b64
        gc.collect()

    for p in page_results:
        if p["method"] == "pending":
            p["method"] = "skipped"
            p["text"] = ""

    fitz.TOOLS.store_shrink(100)
    gc.collect()
    return build_record(meta, page_results, elapsed_s=time.time() - t0)


async def process_chunk(
    chunk: list[dict],
    session: aiohttp.ClientSession,
    pbar,
    out_fh,
    processed_ids: set,
    progress_data: dict,
    counters: dict,
):
    queue = asyncio.Queue()
    for meta in chunk:
        queue.put_nowait(meta)

    lock = asyncio.Lock()
    pause_event = asyncio.Event()
    pause_event.set()
    loop = asyncio.get_event_loop()

    with ThreadPoolExecutor(max_workers=THREAD_WORKERS) as executor:
        async def worker():
            while True:
                await pause_event.wait()

                try:
                    meta = queue.get_nowait()
                except asyncio.QueueEmpty:
                    return

                do_cooldown = False
                cooldown_at = None

                try:
                    record = await asyncio.wait_for(
                        process_pdf_async(meta, session, executor, loop),
                        timeout=PDF_TASK_TIMEOUT_S,
                    )
                    async with lock:
                        out_fh.write(orjson.dumps(record) + b"\n")
                        cid = meta["candidate_id"]
                        if cid not in processed_ids:
                            processed_ids.add(cid)
                            progress_data["processed"].append(cid)
                        counters["written"] += 1
                except Exception as e:
                    msg = str(e)
                    async with lock:
                        counters["errors"] += 1
                        print(f"\nError {meta['pdf_name']}: {msg}")
                    if ("Cannot connect to host" in msg) or ("vLLM HTTP 500" in msg) or ("EngineCore" in msg):
                        raise RuntimeError("fatal_vllm")
                finally:
                    async with lock:
                        counters["processed"] += 1
                        elapsed = time.time() - counters["start"]
                        rate = max(counters["processed"], 1) / max(elapsed, 1)
                        remaining_n = counters["total"] - counters["processed"]
                        eta = remaining_n / max(rate, 0.001) / 60

                        pbar.update(1)
                        pbar.set_postfix({
                            "written": counters["written"],
                            "errors": counters["errors"],
                            "ETA_min": f"{eta:.0f}",
                        })

                        if counters["processed"] % CHECKPOINT_EVERY == 0:
                            out_fh.flush()
                            with open(PROGRESS_FILE, "wb") as pf:
                                pf.write(orjson.dumps(progress_data))

                        # Trigger cooldown once at each threshold (75, 150, 225, ...)
                        if counters["processed"] >= counters["next_cooldown_at"]:
                            do_cooldown = True
                            cooldown_at = counters["next_cooldown_at"]
                            counters["next_cooldown_at"] += COOLDOWN_EVERY_PDFS

                    queue.task_done()

                if do_cooldown:
                    pause_event.clear()
                    out_fh.flush()
                    with open(PROGRESS_FILE, "wb") as pf:
                        pf.write(orjson.dumps(progress_data))
                    print(f"\nCooldown: processed {cooldown_at} PDFs. Sleeping {COOLDOWN_SECONDS}s...")
                    await asyncio.sleep(COOLDOWN_SECONDS)
                    fitz.TOOLS.store_shrink(100)
                    gc.collect()
                    pause_event.set()

        workers = [asyncio.create_task(worker()) for _ in range(CONCURRENT_REQUESTS)]
        await asyncio.gather(*workers)


async def run_ingestion(remaining: list[dict], processed_ids: set, progress_data: dict):
    counters = {
        "written": 0,
        "errors": 0,
        "processed": 0,
        "start": time.time(),
        "total": len(remaining),
        "next_cooldown_at": COOLDOWN_EVERY_PDFS,
    }

    out_fh = open(ENRICHED_JSONL, "ab")
    pbar = tqdm(total=len(remaining), desc="Resumes", unit="pdf")

    try:
        for idx, chunk in enumerate(chunked(remaining, CHUNK_SIZE), start=1):
            connector = aiohttp.TCPConnector(limit=CONCURRENT_REQUESTS + 2)
            async with aiohttp.ClientSession(connector=connector) as session:
                if not await vllm_healthcheck(session):
                    raise RuntimeError("vLLM is not healthy. Restart server and rerun.")

                try:
                    await process_chunk(
                        chunk=chunk,
                        session=session,
                        pbar=pbar,
                        out_fh=out_fh,
                        processed_ids=processed_ids,
                        progress_data=progress_data,
                        counters=counters,
                    )
                except RuntimeError as e:
                    if str(e) == "fatal_vllm":
                        raise RuntimeError("Fatal vLLM failure detected. Stopping safely.")

            out_fh.flush()
            with open(PROGRESS_FILE, "wb") as pf:
                pf.write(orjson.dumps(progress_data))
            fitz.TOOLS.store_shrink(100)
            gc.collect()
            print(f"\nChunk {idx} completed. Checkpoint saved.")
    finally:
        out_fh.flush()
        out_fh.close()
        pbar.close()
        with open(PROGRESS_FILE, "wb") as pf:
            pf.write(orjson.dumps(progress_data))

    elapsed = time.time() - counters["start"]
    print(f"\n{'=' * 60}")
    print("Done.")
    print(f"  Written   : {counters['written']} / {len(remaining)}")
    print(f"  Errors    : {counters['errors']}")
    print(f"  Time      : {elapsed / 60:.1f} min")
    if counters["written"] > 0:
        print(f"  Avg s/pdf : {elapsed / counters['written']:.1f}s")
    print(f"  Output    : {ENRICHED_JSONL}")
    if counters["errors"] > 0:
        print("Failed PDFs were not marked processed; rerun after fixes to retry them.")

print("Async functions defined")

Async functions defined


In [8]:
#Main Ingestion Loop
import orjson
from pathlib import Path

# Load checkpoint
if Path(PROGRESS_FILE).exists():
    with open(PROGRESS_FILE, "rb") as f:
        progress_data = orjson.loads(f.read())
    processed_ids = set(progress_data.get("processed", []))
    print(f"Resuming: {len(processed_ids)} already done")
else:
    progress_data = {"processed": []}
    processed_ids = set()

remaining = [m for m in manifest if m["candidate_id"] not in processed_ids]
print(f"Remaining: {len(remaining)} PDFs")
print(f"Concurrent requests: {CONCURRENT_REQUESTS}")

# Run — asyncio.run() works in scripts; in Jupyter use nest_asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"])
    import nest_asyncio
    nest_asyncio.apply()

asyncio.run(run_ingestion(remaining, processed_ids, progress_data))

Resuming: 1991 already done
Remaining: 781 PDFs
Concurrent requests: 2


Resumes:  13%|█▎        | 100/781 [25:56<4:19:35, 22.87s/pdf, written=100, errors=0, ETA_min=177]


Cooldown: processed 100 PDFs. Sleeping 30s...


Resumes:  19%|█▉        | 150/781 [37:41<2:03:38, 11.76s/pdf, written=150, errors=0, ETA_min=159]


Chunk 1 completed. Checkpoint saved.


...


Cooldown: processed 700 PDFs. Sleeping 30s...


Resumes:  96%|█████████▌| 750/781 [3:09:59<05:31, 10.71s/pdf, written=750, errors=0, ETA_min=8] 


Chunk 5 completed. Checkpoint saved.


Resumes: 100%|██████████| 781/781 [3:16:28<00:00, 15.09s/pdf, written=781, errors=0, ETA_min=0]


Chunk 6 completed. Checkpoint saved.

Done.
  Written   : 781 / 781
  Errors    : 0
  Time      : 196.5 min
  Avg s/pdf : 15.1s
  Output    : ../data/extracted_text/enriched_resumes.jsonl


In [9]:
# Quality Check (streaming, memory-safe)
import orjson
from collections import defaultdict, Counter

total_records = 0
method_counts = Counter()
empty_count = 0
cat_stats = defaultdict(lambda: {"count": 0, "total_chars": 0})

with open(ENRICHED_JSONL, "rb") as f:
    for line in f:
        if not line.strip():
            continue
        r = orjson.loads(line)
        total_records += 1

        meta = r.get("metadata", {})
        total_chars = meta.get("total_chars", 0)
        if total_chars < 50:
            empty_count += 1

        for m in meta.get("methods_used", []):
            method_counts[m] += 1

        cat = meta.get("category", "unknown")
        cat_stats[cat]["count"] += 1
        cat_stats[cat]["total_chars"] += total_chars

print(f"Total records: {total_records}")

print("\nExtraction methods:")
for method, count in method_counts.most_common():
    print(f"  {method:20s} {count:>5}")

print(f"Empty (<50 chars): {empty_count}")

print(f"\n{'Category':45s} {'Count':>6} {'Avg chars':>10}")
print("-" * 65)
for cat, s in sorted(cat_stats.items(), key=lambda x: -x[1]["count"]):
    avg = s["total_chars"] // max(s["count"], 1)
    print(f"{cat:45s} {s['count']:>6} {avg:>10,}")

Total records: 2786

Extraction methods:
  vllm                  2786
Empty (<50 chars): 2

Category                                       Count  Avg chars
-----------------------------------------------------------------
Advocate resumes                                 178      2,384
Apparel resumes                                   92      2,228
Mechanical Engineer resumes                       92      2,533
Accountant resumes                                88      2,606
Arts resumes                                      88      2,313
Consultant resumes                                88      2,755
DevOps Engineer resumes                           84      2,806
Database resumes                                  82      2,321
data science resumes                              80      2,455
Civil Engineer resumes                            79      2,372
Aviation resumes                                  78      2,566
Architects resumes                                76      2,424
Agricultur